In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import box
from matplotlib.patches import Patch

In [ ]:
# setup
path = '~/Desktop/Desktop/epidemiology_PhD/00_repos/la-wf/01_data/01_raw/'

# read in geojson
evac_lp = gpd.read_file(path + 'jan_8_boundaries.geojson')
evac_hmb = gpd.read_file(path + 'california_active_evacuation_zones_20250113_193346.geojson')

# palette
okeefe = ["#fbe3c2", "#f2c88f", "#ecb27d", "#e69c6b", "#d37750", "#b9563f", "#611F10"]


In [ ]:
evac_lp.plot()

In [ ]:
evac_hmb.plot()

In [ ]:
print(f"CRS of evac_lp: {evac_lp.crs}")
print(f"CRS of evac_hmb: {evac_hmb.crs}")

In [ ]:
# clip to logan's bounds because they are specific to the la fires
# use logan's bounds since his is specific to LA fires
bounds_lp = evac_lp.total_bounds
minx, miny, maxx, maxy = bounds_lp

bbox = box(minx, miny, maxx, maxy)
bbox_gdf = gpd.GeoDataFrame({'geometry': [bbox]}, crs=evac_lp.crs)

# Clip evac_hmb to the bounds of evac_lp
evac_hmb_clipped = gpd.clip(evac_hmb, bbox_gdf)

# for contextily basemap, we need to use web mercator projection (EPSG:3857)
if evac_lp.crs != 'EPSG:3857':
    evac_lp_webmerc = evac_lp.to_crs(epsg=3857)
    evac_hmb_webmerc = evac_hmb_clipped.to_crs(epsg=3857)
else:
    evac_lp_webmerc = evac_lp
    evac_hmb_webmerc = evac_hmb_clipped

# make plot
fig, ax = plt.subplots(figsize=(20, 7))

evac_lp_webmerc.plot(ax=ax, color=okeefe[1], alpha=0.7, edgecolor='black', linewidth=0.5, label='LP boundaries')
evac_hmb_webmerc.plot(ax=ax, color=okeefe[5], alpha=0.7, edgecolor='black', linewidth=0.5, label='HMB boundaries')
legend_elements = [
    Patch(facecolor=okeefe[1], edgecolor='black', alpha=0.7, label='LP boundaries'),
    Patch(facecolor=okeefe[5], edgecolor='black', alpha=0.7, label='HMB boundaries')
]

ax.legend(handles=legend_elements, loc='upper left', frameon=True)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
plt.title('evac boundaries: overlay')

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7), sharey=True)

# Get bounds in web mercator for proper display with contextily
bounds_webmerc = evac_lp_webmerc.total_bounds
minx_web, miny_web, maxx_web, maxy_web = bounds_webmerc

# plot 1: LP
evac_lp_webmerc.plot(ax=ax1, color=okeefe[1], alpha=0.7, edgecolor='black', linewidth=0.5)
ctx.add_basemap(ax1, source=ctx.providers.CartoDB.Positron)
ax1.set_title('LP evac zones', fontsize=14)
ax1.set_xlim(minx_web, maxx_web)
ax1.set_ylim(miny_web, maxy_web)
ax1.set_axis_off()

# plot 2: HMB
evac_hmb_webmerc.plot(ax=ax2, color=okeefe[5], alpha=0.7, edgecolor='black', linewidth=0.5)
ctx.add_basemap(ax2, source=ctx.providers.CartoDB.Positron)
ax2.set_title('HMB evac zones', fontsize=14)
ax2.set_xlim(minx_web, maxx_web)  # Fixed: ax2 instead of ax1
ax2.set_ylim(miny_web, maxy_web)  # Fixed: ax2 instead of ax1
ax2.set_axis_off()

plt.suptitle('evac boundaries: side by side', fontsize=20, y=0.95)

plt.tight_layout()
plt.show()